# Chicago Food Inspections — Initial EDA & Feasibility Check

**Capstone:** Predictive Food Safety Intelligence Platform  
**Author:** Jun Xu — UC Berkeley MIDS  
**Notebook goal:** Decide whether the Chicago Food Inspections dataset can support a *predictive*, *explainable* model of inspection risk.

We answer five feasibility questions in order:

1. **Volume** — is there enough labeled data?
2. **Label quality** — is the target (Pass / Fail) clean and balanced enough to model?
3. **History per facility** — do we observe enough repeat inspections per restaurant to learn from? *(This is the critical one — without per-facility history, there is no useful prediction.)*
4. **Signal richness** — do features like prior violations, inspection cadence, facility type, and geography vary meaningfully with outcomes?
5. **Time stability** — does the data look consistent across years (no schema breaks, no COVID gaps that would derail training)?

Data source: City of Chicago open data portal, dataset `4ijn-s7e5` (SODA endpoint).

## 1. Setup

In [ ]:
import io
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# SODA endpoint (more reliable + supports SoQL filtering than the v3 query endpoint).
# Reference: https://dev.socrata.com/foundry/data.cityofchicago.org/4ijn-s7e5
SODA_URL = 'https://data.cityofchicago.org/resource/4ijn-s7e5.json'

## 2. Pull the data

Chicago has ~280k inspections going back to 2010. For an EDA we pull the last ~6 years (covers pre-COVID, COVID, and post-COVID) which keeps the dataset modeling-relevant without being unwieldy. SODA caps at 50k rows per request so we paginate.

In [ ]:
def fetch_inspections(since='2019-01-01', page_size=50_000, max_pages=20):
    """Paginate the SODA endpoint and return a single DataFrame.

    We sort by inspection_date so pagination is deterministic — without an
    explicit $order, SODA does not guarantee stable ordering across pages.
    """
    frames = []
    for page in range(max_pages):
        params = {
            '$where': f"inspection_date >= '{since}T00:00:00'",
            '$order': 'inspection_date ASC, inspection_id ASC',
            '$limit': page_size,
            '$offset': page * page_size,
        }
        r = requests.get(SODA_URL, params=params, timeout=60)
        r.raise_for_status()
        chunk = pd.DataFrame(r.json())
        if chunk.empty:
            break
        frames.append(chunk)
        print(f'  page {page+1}: {len(chunk):,} rows (cumulative {sum(len(f) for f in frames):,})')
        if len(chunk) < page_size:
            break
    return pd.concat(frames, ignore_index=True)

raw = fetch_inspections(since='2019-01-01')
print(f'\nTotal rows pulled: {len(raw):,}')

## 3. Clean & coerce types

Everything comes back as strings from SODA. Cast dates, lat/lon, and inspection_id, then drop the Socrata-computed region columns (we'd derive better geo features ourselves from lat/lon).

In [ ]:
df = raw.copy()

# Drop Socrata-internal computed columns — noisy and not portable.
df = df.loc[:, ~df.columns.str.startswith(':@computed_region')]
df = df.drop(columns=[c for c in ['location'] if c in df.columns])

# Type coercion
df['inspection_date'] = pd.to_datetime(df['inspection_date'])
df['inspection_id'] = pd.to_numeric(df['inspection_id'], errors='coerce').astype('Int64')
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# license_ is sometimes '0' or missing — keep as string for stable grouping
df['license_'] = df['license_'].fillna('').astype(str)

print('Shape:', df.shape)
print('Date range:', df['inspection_date'].min().date(), '→', df['inspection_date'].max().date())
df.dtypes

In [ ]:
# Missingness — flag fields we'll need to be defensive about during feature engineering.
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].to_frame('frac_missing').style.format('{:.2%}')

## 4. Feasibility Q1 — Volume

How many inspections per year? Modeling needs roughly thousands of labeled events per year to learn temporal patterns.

In [ ]:
df['year'] = df['inspection_date'].dt.year
df['month'] = df['inspection_date'].dt.to_period('M')

per_year = df.groupby('year').size().rename('inspections')
print(per_year.to_string())

ax = df.groupby('month').size().plot(title='Inspections per month (Chicago)')
ax.set_ylabel('count')
ax.set_xlabel('')
plt.tight_layout(); plt.show()

**Read:** Watch for the COVID-19 dip in 2020 — inspections paused. This is important: any model trained naively across that gap will see distribution shift. We'll likely either (a) exclude Mar–Dec 2020 from training or (b) add a COVID-era indicator feature.

## 5. Feasibility Q2 — Label distribution

The `results` field is our target. Real outcomes are Pass / Pass w/ Conditions / Fail. The others (Out of Business, No Entry, Not Ready, Business Not Located) are operational non-outcomes — they have to be filtered out before modeling, otherwise we'd be predicting whether the inspector found the business open, not food safety.

In [ ]:
results_counts = df['results'].value_counts(dropna=False)
print(results_counts)
print('\n% share:')
print((results_counts / len(df) * 100).round(2))

In [ ]:
# Define the modeling target: binary Fail vs Pass-ish, excluding non-outcomes.
MODELABLE_RESULTS = {'Pass', 'Pass w/ Conditions', 'Fail'}
modelable = df[df['results'].isin(MODELABLE_RESULTS)].copy()
modelable['fail'] = (modelable['results'] == 'Fail').astype(int)

print(f'Modelable rows: {len(modelable):,} ({len(modelable)/len(df):.1%} of pull)')
print(f'Fail rate:      {modelable["fail"].mean():.2%}')

# Fail rate over time — concept drift check.
ax = modelable.groupby(modelable['inspection_date'].dt.to_period('Q'))['fail'].mean().plot(
    title='Fail rate per quarter (concept drift check)', marker='o')
ax.set_ylabel('P(Fail)'); ax.set_xlabel('')
plt.tight_layout(); plt.show()

**Read:** A ~15–25% fail rate is healthy for binary classification — not so rare we need extreme resampling, not so balanced that the problem is trivial. A flat-ish quarterly fail rate means the target is stationary enough to model.

## 6. Feasibility Q3 — History per facility (the critical question)

The whole pitch — *"is this restaurant becoming riskier over time?"* — only works if most restaurants have multiple inspections. We group by `license_` (more stable than name) and count.

In [ ]:
# Drop license_='0' and empty — these are placeholder licenses that pool across facilities.
valid_lic = modelable[(modelable['license_'] != '') & (modelable['license_'] != '0')]

per_facility = valid_lic.groupby('license_').size()
print('Facilities (unique licenses):', f'{per_facility.size:,}')
print('\nInspections per facility (distribution):')
print(per_facility.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(1))

ax = per_facility.clip(upper=20).plot.hist(bins=20, title='Inspections per facility (clipped at 20)')
ax.set_xlabel('# inspections'); plt.tight_layout(); plt.show()

share_with_history = (per_facility >= 2).mean()
print(f'\nShare of facilities with ≥2 inspections: {share_with_history:.1%}')
print(f'Share with ≥4 inspections:                 {(per_facility >= 4).mean():.1%}')

**Read:** If a strong majority of facilities have ≥2 inspections, we can build per-facility features like *prior fail count*, *days since last inspection*, *trailing violation rate*. That is the backbone of the Chicago precedent model (Copel et al., 2014).

Below we compute the actual time-between-inspections distribution — it tells us the inspection cadence and informs the prediction horizon.

In [ ]:
# For facilities with ≥2 inspections, compute gaps.
sorted_insp = valid_lic.sort_values(['license_', 'inspection_date'])
sorted_insp['days_since_prev'] = (
    sorted_insp.groupby('license_')['inspection_date'].diff().dt.days
)
gap_stats = sorted_insp['days_since_prev'].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
print('Days between consecutive inspections at the same facility:')
print(gap_stats.round(1))

ax = sorted_insp['days_since_prev'].clip(upper=730).plot.hist(
    bins=40, title='Time between consecutive inspections (clipped 2y)')
ax.set_xlabel('days'); plt.tight_layout(); plt.show()

## 7. Feasibility Q4 — Feature signal

Spot-check whether obvious features actually correlate with the target. If e.g. risk-tier and facility-type don't move fail rates at all, the modeling job is much harder.

In [ ]:
def fail_rate_by(col, min_n=200):
    """Fail rate broken out by a categorical, filtered to groups with enough data."""
    grp = modelable.groupby(col).agg(n=('fail', 'size'), fail_rate=('fail', 'mean'))
    return grp[grp['n'] >= min_n].sort_values('fail_rate', ascending=False)

print('--- Fail rate by risk tier ---')
print(fail_rate_by('risk'))
print('\n--- Fail rate by inspection type (top 10) ---')
print(fail_rate_by('inspection_type').head(10))
print('\n--- Fail rate by facility type (top 10) ---')
print(fail_rate_by('facility_type').head(10))

In [ ]:
# Prior-fail signal: does a facility's *previous* fail count predict the next outcome?
# sorted_insp already carries `fail` (inherited from modelable via valid_lic) — no merge needed.
# Leak-free prior count = inclusive per-group cumsum minus the current row's own value.
# (groupby().cumsum().shift() looks right but shift() is not group-aware, so it would
#  bleed the last value of license A into the first row of license B.)
sorted_insp['prior_fails'] = (
    sorted_insp.groupby('license_')['fail'].cumsum() - sorted_insp['fail']
)
sorted_insp['prior_inspections'] = sorted_insp.groupby('license_').cumcount()

by_prior = sorted_insp[sorted_insp['prior_inspections'] >= 1].copy()
by_prior['prior_fails_bucket'] = by_prior['prior_fails'].clip(upper=3).astype(int)
print('Fail rate by # prior fails (leak-free):')
print(by_prior.groupby('prior_fails_bucket').agg(
    n=('fail', 'size'), fail_rate=('fail', 'mean')
).round(3))

**Read:** If fail rate climbs monotonically with prior-fail count, we have a real signal — and a baseline beating logistic-regression model is realistic.

## 8. Violations text — richness for NLP / feature engineering

The `violations` field is the most underused part of this dataset. It's a `|`-separated list of violations, each prefixed with a numbered code (e.g. `10. ADEQUATE HANDWASHING SINKS...`). Counting per-inspection violations and extracting codes gives us cheap structured features before any NLP.

In [ ]:
VIOLATION_CODE_RE = re.compile(r'(?:^|\|)\s*(\d{1,2})\.\s')

def violation_codes(text):
    if not isinstance(text, str):
        return []
    return [int(c) for c in VIOLATION_CODE_RE.findall(text)]

modelable['violation_codes'] = modelable['violations'].apply(violation_codes)
modelable['n_violations'] = modelable['violation_codes'].str.len()

print('Violation count distribution:')
print(modelable['n_violations'].describe(percentiles=[0.5, 0.9, 0.99]).round(1))

# Top violation codes — Chicago numbers 1–63; 1–29 are priority/priority-foundation
# (the more serious tier), 30+ are core. So a high count in the low-numbered codes
# is the meaningful risk signal.
all_codes = pd.Series([c for codes in modelable['violation_codes'] for c in codes])
print('\nTop 15 violation codes:')
print(all_codes.value_counts().head(15))

In [ ]:
# Is a 'priority' violation (code 1–29) predictive of a Fail outcome?
modelable['has_priority_violation'] = modelable['violation_codes'].apply(
    lambda codes: any(c <= 29 for c in codes)
)
print('Fail rate by presence of priority (1–29) violation:')
print(modelable.groupby('has_priority_violation')['fail'].agg(['mean', 'size']).round(3))

## 9. Geography — does location carry signal?

A quick lat/lon scatter colored by fail rate per zip. If hotspots exist we can layer in 311 complaint density and neighborhood features later.

In [ ]:
by_zip = modelable.groupby('zip').agg(
    n=('fail', 'size'),
    fail_rate=('fail', 'mean'),
    lat=('latitude', 'mean'),
    lon=('longitude', 'mean'),
).dropna()
by_zip = by_zip[by_zip['n'] >= 200]

fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(by_zip['lon'], by_zip['lat'], c=by_zip['fail_rate'],
                s=by_zip['n'] / 10, cmap='RdYlGn_r', edgecolor='white', linewidth=0.4)
ax.set_title('Fail rate by ZIP (size = inspection volume)')
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
plt.colorbar(sc, ax=ax, label='P(Fail)')
plt.tight_layout(); plt.show()

print('Highest-fail-rate ZIPs (min 200 inspections):')
print(by_zip.sort_values('fail_rate', ascending=False).head(10).round(3))

## 10. Feasibility verdict

Cell below stitches the answers together. Fill in any caveats discovered above.

In [ ]:
summary = {
    'rows_pulled': len(df),
    'modelable_rows': len(modelable),
    'date_range': f"{df['inspection_date'].min().date()} → {df['inspection_date'].max().date()}",
    'fail_rate': round(modelable['fail'].mean(), 4),
    'unique_facilities': int(per_facility.size),
    'pct_facilities_with_history': round(share_with_history * 100, 1),
    'median_days_between_inspections': float(sorted_insp['days_since_prev'].median()),
    'mean_violations_per_inspection': round(modelable['n_violations'].mean(), 2),
}
pd.Series(summary).to_frame('value')

### Decision criteria — what "feasible" looks like

| Question | Threshold for go | Where we land |
|---|---|---|
| Volume | ≥ 20k modelable inspections/yr | fill in from §4 |
| Label balance | Fail rate 10–40% | fill in from §5 |
| Per-facility history | ≥ 70% of facilities have ≥2 inspections | fill in from §6 |
| Signal | Prior-fail features move P(Fail) by ≥ 5pp | fill in from §7 |
| Time stability | No multi-quarter schema gaps outside COVID | fill in from §4 |

### Likely next steps (week 2)

1. **Materialize the modeling table** — one row per (facility, inspection_date), with leak-free features: prior fail count, prior priority-violation count, days-since-last-inspection, facility type, risk tier, zip, season.
2. **Train baseline** — logistic regression first, then gradient boosting (LightGBM). Holdout by *time*, not by random split — that's how the deployed model will run.
3. **Evaluate at precision@K** — the metric that matters for inspector prioritization ("of the top 100 restaurants we flag this month, how many actually fail?").
4. **Layer in 311 + business-license data** — neighborhood sanitation complaints, license-age, license-type are the most promising external signals.
5. **Stand up explainability** — SHAP on the LightGBM model so every risk score comes with a 'why' breakdown for the consumer / inspector UI.